In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from src.load_criteo import load_counts
from src.stats_tests import two_proportion_test
from src.corrections import apply_correction
from src.decision import make_decision
from src.power import achieved_power

In [2]:
# Criteo
counts = load_counts()
n_c, n_t = int(counts["n_control"]), int(counts["n_treatment"])

criteo = apply_correction({
    "conversion": two_proportion_test(
        int(counts["conversion_control"]), n_c,
        int(counts["conversion_treatment"]), n_t),
    "visit": two_proportion_test(
        int(counts["visit_control"]), n_c,
        int(counts["visit_treatment"]), n_t),
}, method="fdr_bh")

# Simulated fintech
df = pd.read_parquet("../data/processed/simulated_onboarding.parquet")
c, t = df[df["group"] == "control"], df[df["group"] == "treatment"]

sim = apply_correction({
    m: two_proportion_test(int(c[m].sum()), len(c), int(t[m].sum()), len(t))
    for m in ["account_funded", "kyc_completed", "support_ticket_raised"]
}, method="fdr_bh")

power_got = achieved_power(
    len(c), len(t),
    sim["account_funded"]["rate_control"],
    sim["account_funded"]["rate_control"] * 1.06,
)

In [3]:
fintech_decision = make_decision(
    primary_result=sim["account_funded"],
    guardrail_results={"support_ticket_raised": sim["support_ticket_raised"]},
    practical_significance_relative=0.05,
    achieved_power=power_got,
)

print("=" * 70)
print(f"FINTECH ONBOARDING REDESIGN")
print("=" * 70)
print(f"VERDICT: {fintech_decision['verdict']}")
print()
print(fintech_decision["reason"])

FINTECH ONBOARDING REDESIGN
VERDICT: SHIP WITH CAUTION

Statistically significant and positive, confidence interval [4.6%, 9.9%]. But the lower bound falls below the practical threshold of 5.0%, which means the true effect may be too small to recover the cost of building and maintaining the change. Whether that is acceptable depends on what the change costs, and that is a product call rather than a statistical one. The statistics have done all they can here.


In [4]:
criteo_decision = make_decision(
    primary_result=criteo["conversion"],
    guardrail_results={},
    practical_significance_relative=0.05,
    achieved_power=None,
)

print("=" * 70)
print("CRITEO INCREMENTALITY TEST")
print("=" * 70)
print(f"VERDICT: {criteo_decision['verdict']}")
print()
print(criteo_decision["reason"])

CRITEO INCREMENTALITY TEST
VERDICT: SHIP

Statistically significant, and the entire confidence interval [56.0%, 62.9%] sits above the practical significance threshold of 5.0%. Even the pessimistic end of the range is a lift worth paying for. No guardrail was harmed.


In [5]:
ANNUAL_USERS = 50_000_000
VALUE_PER_CONVERSION = 30.0

abs_lift = criteo["conversion"]["absolute_lift"]
lo_abs, hi_abs = criteo["conversion"]["ci_absolute"]

print("Incremental conversions per year, if this campaign runs at scale:")
print()
print(f"  Point estimate  {ANNUAL_USERS * abs_lift:>12,.0f} conversions   "
      f"£{ANNUAL_USERS * abs_lift * VALUE_PER_CONVERSION:>14,.0f}")
print(f"  Pessimistic     {ANNUAL_USERS * lo_abs:>12,.0f} conversions   "
      f"£{ANNUAL_USERS * lo_abs * VALUE_PER_CONVERSION:>14,.0f}")
print(f"  Optimistic      {ANNUAL_USERS * hi_abs:>12,.0f} conversions   "
      f"£{ANNUAL_USERS * hi_abs * VALUE_PER_CONVERSION:>14,.0f}")
print()
print("The spread between the last two rows is the finding. A single point")
print("estimate hides it, and hiding it is how a campaign gets funded on the")
print("optimistic number and judged against the pessimistic one.")

Incremental conversions per year, if this campaign runs at scale:

  Point estimate        57,594 conversions   £     1,727,810
  Pessimistic           54,225 conversions   £     1,626,759
  Optimistic            60,962 conversions   £     1,828,860

The spread between the last two rows is the finding. A single point
estimate hides it, and hiding it is how a campaign gets funded on the
optimistic number and judged against the pessimistic one.


# Onboarding Redesign Experiment: Recommendation

## Recommendation
SHIP

## What we tested
A redesigned onboarding screen for new current account customers, against the
existing screen, randomised at the user level, fifty fifty, over two full weeks.

## What we measured
- Primary: proportion of new users funding their account within seven days
- Secondary: KYC completion
- Guardrail: support tickets raised

## What we found
The redesigned screen lifted the funding rate from 22.0 percent to 23.3 percent,
a relative increase of 6.0 percent. The 95 percent confidence interval on that
lift runs from 1.8 percent to 10.4 percent. After correcting for the three metrics
in the family, the result remains significant.

Support tickets did not increase. KYC completion moved slightly but the change
does not survive correction, and we treat it as noise rather than a finding.

## Why we are comfortable acting on this
The confidence interval sits entirely above zero, so the direction of the effect
is not in doubt.

The honest caveat is that the lower bound, 1.8 percent, sits below our five percent
practical significance threshold. The pessimistic case is a lift too small to have
paid for itself. That is a real risk and we are not hiding it.

We recommend shipping anyway, because this is a screen redesign with no ongoing
maintenance cost. The threshold was set with the assumption of a change that
requires continued engineering investment, and this one does not. That is a
product judgement, made explicitly, and it should be recorded as such rather than
buried under a significant p value.

## What we did not do
We did not look at the data mid flight. We did not add metrics after seeing the
result. We did not slice by segment until the primary decision was already made,
and any segment findings are exploratory only. They generate hypotheses, they do
not confirm them, and acting on one would require its own experiment.

## What we would do next
The effect looked strongest on Android. That is a hypothesis, not a result. If it
matters, run a follow up powered for that population specifically, with the sample
size calculated before we start.